# Día 12 · Arquitectura multiagente sencilla

Este notebook verifica el grafo coordinado de cuatro nodos sin usar credenciales ni evidencia privada. La conexión con los adaptadores operativos se realizará en el Día 13.

In [ ]:
!pip -q install 'langgraph>=0.2,<2'

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / 'src').exists():
    raise RuntimeError('Ejecuta el notebook desde la raíz del repositorio')
sys.path.insert(0, str(repo))

from src.agents.risk_graph import RiskGraphDependencies, run_risk_graph

## Prueba sintética reproducible

Los adaptadores siguientes simulan únicamente los límites del grafo. No sustituyen los resultados aprobados de los días 1–11.

In [ ]:
def retrieval_demo(question, document_ids):
    return [{'chunk_id': 'DEMO_01', 'score': 0.91, 'chunk_text': 'Fragmento sintético'}]

def extraction_demo(chunks):
    return [{
        'item_id': 'ITEM_DEMO_01', 'source_doc_id': 'DOC_DEMO',
        'source_filename': 'documento_demo.pdf', 'source_page': 1,
        'source_chunk_id': 'DEMO_01', 'title': 'Retraso documentado',
        'statement': 'Existe un retraso pendiente.',
        'evidence_quote': 'El entregable continúa pendiente después de la fecha acordada.',
        'evidence_verified': 1, 'calibrated_type': 'HECHO_OCURRIDO',
        'calibrated_watch': 1, 'calibrated_evidence_sufficient': 1,
        'calibrated_category': 'Cronograma', 'calibrated_confidence': 0.9,
        'calibrated_justification': 'La evidencia confirma el retraso.'
    }]

def profile_demo(signals):
    return {'profile_status': 'PROVISIONAL', 'signals_total': len(signals)}

deps = RiskGraphDependencies(retrieval_demo, extraction_demo, profile_demo)
result = run_risk_graph(deps, '¿Qué retrasos requieren vigilancia?', request_id='DEMO-DIA-12')
result['status'], result['execution_trace'], result['risk_profile']

## Criterio de aceptación

La ejecución debe terminar en `COMPLETED` y la traza debe contener, en orden: `retrieval`, `risk_extraction`, `risk_validation` y `risk_profile`.